## Path setup and load everything

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import attacks
import evaluation

import numpy as np
import torch
import joblib

# Load the preprocessed arrays and fitted objects from Stage 2.
data = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
label_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
class_names = list(label_encoder.classes_)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded. X_test: {X_test.shape}  device: {device}")

Loaded. X_test: (718, 9)  device: cuda


## Retrain the baseline CNN

In [2]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights as same as stage 3
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(w, dtype=torch.float32, device=device)

# Train the baseline CNN on the full duplicated training set
cnn = models.CNN1D(n_features=9, n_classes=6)
cnn = models.train_cnn(
    cnn, X_train, y_train,
    n_epochs=50, device=device,
    class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("Baseline CNN trained.")

    epoch   1/50     loss 1.6383
    epoch   5/50     loss 0.2470
    epoch  10/50     loss 0.0424
    epoch  15/50     loss 0.0087
    epoch  20/50     loss 0.0041
    epoch  25/50     loss 0.0024
    epoch  30/50     loss 0.0019
    epoch  35/50     loss 0.0014
    epoch  40/50     loss 0.0013
    epoch  45/50     loss 0.0011
    epoch  50/50     loss 0.0010
Baseline CNN trained.


## Wrap CNN and generate FGSM examples

In [3]:
# Wrap the trained CNN
classifier = attacks.wrap_cnn_for_art(cnn, n_features=9, n_classes=6, device=device)

# Baseline: how does the CNN do on the clean test set
cnn.eval()
with torch.no_grad():
    clean_logits = cnn(torch.tensor(X_test, dtype=torch.float32, device=device))
    clean_pred = clean_logits.argmax(dim=1).cpu().numpy()

from sklearn.metrics import f1_score
clean_f1 = f1_score(y_test, clean_pred, average="macro", zero_division=0)
print(f"CNN clean macro-F1 (reference): {clean_f1:.4f}\n")

# Generate FGSM adversarial examples from the test set at a moderate epsilon
X_adv_fgsm = attacks.generate_fgsm(classifier, X_test, epsilon=0.10)

# How does the CNN do on the purturbated test set
with torch.no_grad():
    adv_logits = cnn(torch.tensor(X_adv_fgsm, dtype=torch.float32, device=device))
    adv_pred = adv_logits.argmax(dim=1).cpu().numpy()

adv_f1 = f1_score(y_test, adv_pred, average="macro", zero_division=0)
print(f"CNN FGSM macro-F1 (epsilon=0.10): {adv_f1:.4f}")
print(f"Degredation: {clean_f1 - adv_f1:.4f} drop")

CNN clean macro-F1 (reference): 0.7108

CNN FGSM macro-F1 (epsilon=0.10): 0.1597
Degredation: 0.5511 drop


## Epsilon sweep, FGSM and PGD, against both models

In [4]:
from sklearn.metrics import f1_score

# Helper: get macro-F1 for a model's predictions on some inputs
def cnn_macro_f1(X):
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(dim=1).cpu().numpy()
    return f1_score(y_test, pred, average="macro", zero_division=0)

def rf_macro_f1(X):
    return f1_score(y_test, rf.predict(X), average="macro", zero_division=0)

# Retrain the RF so the notebook is self-contained
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)
rf.fit(X_train, y_train)

# Clean baselines
clean_cnn = cnn_macro_f1(X_test)
clean_rf = rf_macro_f1(X_test)
print(f"CLEAN   CNN={clean_cnn:.4f}     RF={clean_rf:.4f}\n")

# Sweep over the epsilon list from config for both attacks
epsilons = config.FGSM_EPSILONS

results = {"eps": [], "fgsm_cnn": [], "fgsm_rf": [], "pgd_cnn": [], "pgd_rf": []}

for eps in epsilons:
    # FGSM at this epsilon crafted on the cnn
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # PGD at this epsilon crafted on the cnn
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)

    # Feed same cradfted frames to both models (transfer attack)
    results["eps"].append(eps)
    results["fgsm_cnn"].append(cnn_macro_f1(Xf))
    results["fgsm_rf"].append(rf_macro_f1(Xf))
    results["pgd_cnn"].append(cnn_macro_f1(Xp))
    results["pgd_rf"].append(rf_macro_f1(Xp))

    print(f"eps={eps:.2f} | FGSM: CNN={results['fgsm_cnn'][-1]:.4f} RF={results['fgsm_rf'][-1]:.4f}"
          f" | PGD: CNN={results['pgd_cnn'][-1]:.4f} RF={results['pgd_rf'][-1]:.4f}")

CLEAN   CNN=0.7108     RF=0.7761



PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.01 | FGSM: CNN=0.6587 RF=0.5386 | PGD: CNN=0.6587 RF=0.5943


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.05 | FGSM: CNN=0.3759 RF=0.1609 | PGD: CNN=0.3652 RF=0.1599


...

eps=0.10 | FGSM: CNN=0.1597 RF=0.1541 | PGD: CNN=0.1556 RF=0.1495


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.20 | FGSM: CNN=0.0846 RF=0.1605 | PGD: CNN=0.0805 RF=0.1530


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.30 | FGSM: CNN=0.1079 RF=0.1656 | PGD: CNN=0.0799 RF=0.1523


## Verify the perturbation and break down per-class

In [5]:
from sklearn.metrics import classification_report

# Part 1: is the perturbation real at eps=0.01
X_adv_001 = attacks.generate_fgsm(classifier, X_test, epsilon=0.01)

# How much did the inputs actually change?
diff = np.abs(X_adv_001 - X_test)
print("PERTURBATION CHECK at eps=0.01")
print(f"    max abs change per feature  : {diff.max():.5f}")
print(f"    mean abs change             : {diff.mean():.5f}")
print(f"    rows that changed at all    : {(diff.sum(axis=1) > 0).sum()} / {len(X_test)}")

# Did the CNN predictions actually change?
cnn.eval()
with torch.no_grad():
    p_clean = cnn(torch.tensor(X_test, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    p_adv = cnn(torch.tensor(X_adv_001, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
print(f"    CNN predictions changed     : {(p_clean != p_adv).sum()} / {len(X_test)}")

# Part 2: per-class breakdown, RF under FGSM eps=0.01
print("\nRF per-class CLEAN:")
print(classification_report(y_test, rf.predict(X_test), target_names=class_names, zero_division=0))
print("\nRF per-class under FGSM eps=0.01:")
print(classification_report(y_test, rf.predict(X_adv_001), target_names=class_names, zero_division=0))

PERTURBATION CHECK at eps=0.01
    max abs change per feature  : 0.01000
    mean abs change             : 0.00799
    rows that changed at all    : 718 / 718
    CNN predictions changed     : 3 / 718

RF per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718


RF per-class under FGSM eps=0.01:
                         precision    recall  f1-score   support

                    DoS 

## Integer-valid realism check

In [6]:
# Load the fitted scaler to map [0,1] with real CAN byte space (0-255)
scaler = joblib.load(config.PROCESSED_DIR / "feature_scaler.joblib")

def round_to_valid_can(X_scaled):
    """
    Round adversarial frames to legal integer CAN bytes, then re-scale.

    Steps: inverse transform [0,1] -> original 0-255 space, round to nearest integer, clip to 0-255 and then re-scale back to [0-1] for the models.
    The result is a frame an attacker could actually transmit on the bus.
    """
    X_real = scaler.inverse_transform(X_scaled)
    X_real = np.clip(np.round(X_real), config.FEATURE_MIN, config.FEATURE_MAX)
    X_valid = scaler.transform(X_real)
    return X_valid


print("INTEGER-VALID REALISM CHECK (FGSM)\n")
print(f"{'eps':>6} | {'continuous':>22} | {'integer-valid':>22}")
print(f"{'':>6} | {'CNN':>10} {'RF':>10} | {'CNN':>10} {'RF':>10}")

for eps in config.FGSM_EPSILONS:
    # Continuous attack as before
    Xc = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # Integer-valid version of the same attack
    Xv = round_to_valid_can(Xc)

    row = (cnn_macro_f1(Xc), rf_macro_f1(Xc), cnn_macro_f1(Xv), rf_macro_f1(Xv))
    print(f"{eps:>6.2f} | {row[0]:>10.4f} {row[1]:>10.4f} | {row[2]:>10.4f} {row[3]:>10.4f}")

INTEGER-VALID REALISM CHECK (FGSM)

   eps |             continuous |          integer-valid
       |        CNN         RF |        CNN         RF
  0.01 |     0.6587     0.5386 |     0.4267     0.1641
  0.05 |     0.3759     0.1609 |     0.2754     0.1608
  0.10 |     0.1597     0.1541 |     0.0837     0.1608
  0.20 |     0.0846     0.1605 |     0.0985     0.1460
  0.30 |     0.1079     0.1656 |     0.0989     0.1459


## CNN per-class breakdown under attack

In [9]:
# It is confirmed that the CNN is untouched at eps=0.01.
# the interesting CNN collapse happen at eps=0.05.
# Therefore, let's see which classes fail there, to mirror the RF per-class analysis

X_adv_05 = attacks.generate_fgsm(classifier, X_test, epsilon=0.05)

cnn.eval()
with torch.no_grad():
    cnn_pred_05 = cnn(torch.tensor(X_adv_05, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
print("CNN per-class CLEAN:")
print(classification_report(y_test, clean_pred, target_names=class_names, zero_division=0))
print("\nCNN per-class under FGSM eps=0.05:")
print(classification_report(y_test, cnn_pred_05, target_names=class_names, zero_division=0))

CNN per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       0.67      1.00      0.80         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy                           0.99       718
              macro avg       0.64      0.83      0.71       718
           weighted avg       1.00      0.99      0.99       718


CNN per-class under FGSM eps=0.05:
                         precision    recall  f1-score   support

                    DoS       0.02      0.25      0.04         4
                 benign       1.00      0.78      0.88       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM      

## Robust-support metric and the dual-metric sweep

In [10]:
from sklearn.metrics import f1_score

ROBUST_LABELS = [0, 1, 3]       # DoS, benign, spoofing-RPM

def macro_f1_full(y_true, y_pred):
    """6-class macro-F1"""
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def macro_f1_robust(y_true, y_pred):
    """Macro-F1 over only the robust-support classes"""
    return f1_score(y_true, y_pred, labels=ROBUST_LABELS, average="macro", zero_division=0)

def cnn_pred(X):
    cnn.eval()
    with torch.no_grad():
        return cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
# Regenerate the FGSM sweep reporting both metrics for both models
print("DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)\n")
print(f"{'eps':>6} | {'CNN full':>9} {'CNN rob':>8} | {'RF full':>9} {'RF rob':>8}")

clean_cp = cnn_pred(X_test); clean_rp = rf.predict(X_test)
print(f"{'clean':>6} | {macro_f1_full(y_test,clean_cp):>9.4f} {macro_f1_robust(y_test,clean_cp):>8.4f} "
      f"| {macro_f1_full(y_test,clean_rp):>9.4f} {macro_f1_robust(y_test,clean_rp):>8.4f}")

for eps in config.FGSM_EPSILONS:
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    cp = cnn_pred(Xf); rp = rf.predict(Xf)
    print(f"{eps:>6.2f} | {macro_f1_full(y_test,cp):>9.4f} {macro_f1_robust(y_test,cp):>8.4f} "
          f"| {macro_f1_full(y_test,rp):>9.4f} {macro_f1_robust(y_test,rp):>8.4f}")

DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)

   eps |  CNN full  CNN rob |   RF full   RF rob
 clean |    0.7108   0.8660 |    0.7761   0.8855
  0.01 |    0.6587   0.7618 |    0.5386   0.7439
  0.05 |    0.3759   0.4184 |    0.1609   0.3218
  0.10 |    0.1597   0.3193 |    0.1541   0.3081
  0.20 |    0.0846   0.1693 |    0.1605   0.3211
  0.30 |    0.1079   0.2157 |    0.1656   0.3312
